# Investment Decision & Management Dashboard

## Objective

This notebook extends the movie business analytics project into a management decision-support layer. It focuses on outlier-aware investment analysis, investment scoring, capital allocation scenarios, and dashboard-ready business insights.

In [1]:
import pandas as pd

# Load movie dataset
movies_raw = pd.read_csv("tmdb_5000_movies.csv")

# Keep movies with valid budget and revenue
financial_df = movies_raw[
    (movies_raw["budget"] > 0) &
    (movies_raw["revenue"] > 0)
].copy()

# Calculate profit and ROI
financial_df["profit"] = (
    financial_df["revenue"] - financial_df["budget"]
)

financial_df["roi"] = (
    financial_df["profit"] / financial_df["budget"]
)

print("Financial records available:", len(financial_df))

Financial records available: 3229


## Outlier-Aware Investment Analysis

In [2]:
# ============================================================
# OUTLIER-AWARE INVESTMENT ANALYSIS
# ============================================================

strategic_financial = financial_df[
    financial_df["budget"] >= 10_000_000
].copy()

strategic_financial["profit_margin"] = (
    strategic_financial["profit"] / strategic_financial["revenue"]
)

strategic_summary = pd.DataFrame({
    "metric": [
        "Average ROI - All Movies",
        "Median ROI - All Movies",
        "Average ROI - Strategic Budget Segment",
        "Median ROI - Strategic Budget Segment",
        "Profitability Rate - Strategic Budget Segment"
    ],
    "value": [
        financial_df["roi"].mean(),
        financial_df["roi"].median(),
        strategic_financial["roi"].mean(),
        strategic_financial["roi"].median(),
        (strategic_financial["profit"] > 0).mean()
    ]
})

strategic_summary

,metric,value
0,Average ROI - All Movies,2953.822408
1,Median ROI - All Movies,1.300366
2,Average ROI - Strategic Budget Segment,2.083735
3,Median ROI - Strategic Budget Segment,1.149306
4,Profitability Rate - Strategic Budget Segment,0.754105


## Investment Scoring Model

In [3]:
# ============================================================
# INVESTMENT SCORING MODEL
# ============================================================

# Rank-based scores reduce the influence of extreme values
strategic_financial["profit_score"] = (
    strategic_financial["profit"].rank(pct=True) * 100
)

strategic_financial["roi_score"] = (
    strategic_financial["roi"].rank(pct=True) * 100
)

strategic_financial["margin_score"] = (
    strategic_financial["profit_margin"].rank(pct=True) * 100
)

# Weighted investment score
strategic_financial["investment_score"] = (
    0.40 * strategic_financial["profit_score"] +
    0.35 * strategic_financial["roi_score"] +
    0.25 * strategic_financial["margin_score"]
)

# Investment recommendation category
strategic_financial["investment_category"] = pd.cut(
    strategic_financial["investment_score"],
    bins=[-float("inf"), 40, 60, 80, float("inf")],
    labels=[
        "Review / Avoid",
        "Selective Investment",
        "Attractive Investment",
        "Priority Investment"
    ]
)

investment_scorecard = (
    strategic_financial[
        [
            "title",
            "budget",
            "revenue",
            "profit",
            "roi",
            "profit_margin",
            "investment_score",
            "investment_category"
        ]
    ]
    .sort_values("investment_score", ascending=False)
    .head(15)
)

investment_scorecard

,title,budget,revenue,profit,roi,profit_margin,investment_score,investment_category
2967,E.T. the Extra-Terrestrial,10500000,792910554,782410554,74.515291,0.986758,99.535443,Priority Investment
2912,Star Wars,11000000,775398007,764398007,69.490728,0.985814,99.415298,Priority Investment
546,Minions,74000000,1156730962,1082730962,14.631499,0.936027,99.046856,Priority Investment
494,The Lion King,45000000,788241776,743241776,16.516484,0.942911,98.758510,Priority Investment
675,Jurassic Park,63000000,920100000,857100000,13.604762,0.931529,98.694433,Priority Investment
1990,The Empire Strikes Back,18000000,538400000,520400000,28.911111,0.966568,98.470164,Priority Investment
1810,The Passion of the Christ,30000000,611899420,581899420,19.396647,0.950972,98.430116,Priority Investment
506,Despicable Me 2,76000000,970761885,894761885,11.773183,0.921711,98.422107,Priority Investment
0,Avatar,237000000,2787965087,2550965087,10.763566,0.914992,98.414097,Priority Investment
1145,The Sixth Sense,40000000,672806292,632806292,15.820157,0.940548,98.374049,Priority Investment


## Capital Allocation Scenario

In [4]:
# ============================================================
# CAPITAL ALLOCATION SCENARIO
# ============================================================

# Hypothetical investment capital available
capital_pool = 500_000_000

allocation_rows = []
capital_used = 0

# Select highest-scoring investments while staying within the capital limit
for _, row in strategic_financial.sort_values(
    "investment_score", ascending=False
).iterrows():

    if capital_used + row["budget"] <= capital_pool:
        allocation_rows.append(row)
        capital_used += row["budget"]

capital_allocation = pd.DataFrame(allocation_rows)

capital_summary = pd.DataFrame({
    "metric": [
        "Available Capital",
        "Capital Allocated",
        "Remaining Capital",
        "Selected Investments",
        "Historical Revenue",
        "Historical Profit"
    ],
    "value": [
        capital_pool,
        capital_used,
        capital_pool - capital_used,
        len(capital_allocation),
        capital_allocation["revenue"].sum(),
        capital_allocation["profit"].sum()
    ]
})

capital_summary

,metric,value
0,Available Capital,500000000
1,Capital Allocated,493500000
2,Remaining Capital,6500000
3,Selected Investments,12
4,Historical Revenue,9285822550
5,Historical Profit,8792322550


## Scenario Analysis

In [5]:
# ============================================================
# SCENARIO ANALYSIS
# ============================================================

base_revenue = capital_allocation["revenue"].sum()
base_budget = capital_allocation["budget"].sum()

scenario_analysis = pd.DataFrame({
    "scenario": [
        "Base Case",
        "Downside Case",
        "Upside Case"
    ],
    "revenue": [
        base_revenue,
        base_revenue * 0.90,
        base_revenue * 1.10
    ],
    "budget": [
        base_budget,
        base_budget * 1.10,
        base_budget
    ]
})

scenario_analysis["profit"] = (
    scenario_analysis["revenue"] -
    scenario_analysis["budget"]
)

scenario_analysis["roi"] = (
    scenario_analysis["profit"] /
    scenario_analysis["budget"]
)

scenario_analysis

,scenario,revenue,budget,profit,roi
0,Base Case,9.285823e+09,493500000.0,8.792323e+09,17.816256
1,Downside Case,8.357240e+09,542850000.0,7.814390e+09,14.395119
2,Upside Case,1.021440e+10,493500000.0,9.720905e+09,19.697882


## Tableau Dashboard Dataset

In [6]:
# ============================================================
# TABLEAU DASHBOARD DATASET
# ============================================================

dashboard_df = financial_df.copy()

# Extract release year
dashboard_df["release_date"] = pd.to_datetime(
    dashboard_df["release_date"],
    errors="coerce"
)

dashboard_df["release_year"] = (
    dashboard_df["release_date"].dt.year
)

# Profit margin
dashboard_df["profit_margin"] = (
    dashboard_df["profit"] / dashboard_df["revenue"]
)

# Cap extreme ROI values for clearer dashboard visualization
dashboard_df["roi_capped"] = (
    dashboard_df["roi"].clip(upper=10)
)

# Create broad return/risk categories
dashboard_df["risk_category"] = pd.cut(
    dashboard_df["roi"],
    bins=[
        -float("inf"),
        0,
        1,
        3,
        10,
        float("inf")
    ],
    labels=[
        "Loss",
        "Low Return",
        "Moderate Return",
        "High Return",
        "Very High Return"
    ]
)

# Export dashboard-ready data
dashboard_df.to_csv(
    "movie_business_dashboard_data.csv",
    index=False
)

print("Dashboard dataset created successfully.")
print("Records:", len(dashboard_df))
print("Columns:", len(dashboard_df.columns))

Dashboard dataset created successfully.
Records: 3229
Columns: 26
